## Model Testing and Request Dispatching

In this section, we verify the API's functionality by sending requests to free models.
OpenRouter's free models allow for debugging without incurring costs.

Models used:
- `google/gemma-4-26b-a4b-it:free` — Google's Gemma.
- `mistralai/mistral-7b-instruct:free` — Mistral 7B.

The `test_openrouter` function encapsulates the logic for header generation and response handling.

In [2]:
import os
import json
import requests

In [3]:
# Dynamic path definition
BASE_DIR = os.getcwd()
CONFIG_PATH = os.path.join(BASE_DIR, 'config.json')

def load_config():
    try:
        with open(CONFIG_PATH, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: File {CONFIG_PATH} not found.")
        return None

config = load_config()
api_key = config.get('openrouter_api_key') if config else None

if not api_key or api_key == 'YOUR_OPENROUTER_API_KEY':
    print("❌ Error: openrouter_api_key is not configured in config.json")

In [ ]:
def test_openrouter(prompt, model="google/gemma-4-26b-a4b-it:free"):
    if not api_key:
        return
    
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "X-Title": "Jupyter Test Script"
    }
    
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 200:
            result = response.json()
            content = result['choices'][0]['message']['content']
            print(f"🤖 Response from {model}:\n")
            print(content)
        else:
            print(f"❌ Error: {response.status_code}")
            print(response.text)
    except Exception as e:
        print(f"❌ Error occurred: {e}")

# Test run
print("--- Testing Gemma ---")
test_openrouter("Hi! How are you? Tell a short joke about programmers.", model="google/gemma-4-26b-a4b-it:free")

print("\n--- Testing Mistral ---")
test_openrouter("What is the capital of France?", model="mistralai/mistral-7b-instruct:free")
